# 07 · Evaluator Demo — compare ALL THREE models

**For the examiner.** Hand-pick any number of leaf images; every model classifies them and shows *why* via Grad-CAM.

**How to run:** Run the cells top to bottom. In **Step 2** choose your images, then run **Step 3**.

For each image you get, side by side:
| INPUT (true species) | EfficientNetV2-S | Swin-Small | CBAM-ConvNeXt |
|---|---|---|---|

Each model panel shows its **prediction + confidence**, a **Grad-CAM heat map** of the pixels it used, and a **green ✓ / red ✗** border for correct / incorrect.

In [ ]:
# === Preamble 1/2: environment & GPU report ===
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch installs in the next cell.')

In [ ]:
# === Preamble 2/2: clone-or-pull + install ===
import os, subprocess, sys

REPO_URL = "https://github.com/Kidhurshan/plant-leaf-classifier.git"
REPO_DIR = "/content/plant-leaf-classifier"

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'requirements.txt'], check=True)

from src.utils import sync_repo, gpu_report
sync_repo(); gpu_report()

## Step 1 · Load all three trained models
Loads from saved checkpoints only — no training state involved.

In [ ]:
from src.config import load_config
from src.utils import use_drive_paths
from src.inference import (ModelComparer, load_demo_cache,
                           cache_index_by_species)
from src import viz
import torch

cfg = load_config('configs/default.yaml')
use_drive_paths(cfg)          # checkpoints live on Google Drive
# If your checkpoints are elsewhere, set it directly instead:
# cfg.paths.checkpoint_dir = '/content/drive/MyDrive/task4_egypli/checkpoints'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
comparer = ModelComparer(cfg, device)
cache = load_demo_cache(cfg)          # cached images from Drive (no Kaggle needed)
by_species = cache_index_by_species(cache)
print('Models loaded :', list(comparer.models))
print('Species       :', comparer.class_names)
print('Images available:', {k: len(v) for k, v in by_species.items()})

## Step 2 · Choose the images
**Option A — browse the dataset.** Set `SPECIES` and run to see numbered thumbnails, then note the `[index]` values you want.

In [ ]:
SPECIES  = 'apple'   # apple | berry | fig | guava | orange | palm | persimmon | tomato
N_SHOWN  = 24        # how many thumbnails to display

ids = by_species[SPECIES][:N_SHOWN]
viz.plot_contact_sheet([cache['images'][i] for i in ids], labels=ids,
                       title=f'{SPECIES} — pick by the [number] shown');

**Option B — build your selection.** Pick indices from the sheet above, and/or add whole species, and/or paste explicit file paths. Mix freely — there is no limit on how many images you choose.

In [ ]:
# Paste the [numbers] you saw on the sheet (they are cache indices).
SELECTED = [
    # e.g. 3, 17, 42,
]

# Or add every image of a species:  SELECTED += by_species['tomato'][:5]

if not SELECTED:
    SELECTED = [by_species[s][0] for s in by_species]   # one of each species
    print('No manual pick given -> defaulting to one image per species.')
print(f'{len(SELECTED)} image(s) selected:', SELECTED)

*Shortcut:* to grab a random spread across every species instead, run this cell (otherwise skip it).

In [ ]:
import random
PER_CLASS = 2      # random images per species
random.seed()      # fresh pick each run
SELECTED = [random.choice(v) for k, v in by_species.items() for _ in range(PER_CLASS)]
print(f'{len(SELECTED)} random image(s) across {len(by_species)} species:', SELECTED)

## Step 3 · Classify with all three models
Each row = one image. Green ✓ = correct, red ✗ = wrong. The heat map shows which pixels drove that model's decision.

In [ ]:
results = comparer.compare_from_cache(cache, SELECTED, top_k=3, with_gradcam=True)

viz.plot_model_comparison_panel(
    results,
    out_path=f'{cfg.paths.figures_dir}/evaluator_demo_panel.png');

### Per-image detail (top-3 probabilities from every model)

In [ ]:
for r in results:
    print('=' * 72)
    print(f"{r['path'].split('/')[-1]}   true: {r['true'] or 'unknown'}"
          f"   {'ALL AGREE' if r['agreement'] else 'MODELS DISAGREE'}")
    for k, m in r['models'].items():
        flag = '' if m['correct'] is None else ('OK ' if m['correct'] else 'X  ')
        top3 = ', '.join(f'{n} {p*100:.1f}%' for n, p in m['topk'])
        print(f"  {flag}{viz.display_name(k):<18} -> {m['pred']:<10}"
              f" {m['confidence']*100:5.1f}%   | top-3: {top3}")

### Scoreboard

In [ ]:
from src.inference import comparison_summary
s = comparison_summary(results)
print(f"Images reviewed : {s['_n_images']}")
print(f"All-model agreement: {s['_agreement']*100:.0f}%\n")
for k in comparer.models:
    d = s[k]
    print(f"  {viz.display_name(k):<18} {d['correct']}/{d['total']} correct"
          f"  ({d['accuracy']*100:.1f}%)")

## Optional · Test with your own photo
Upload any leaf image (true species unknown, so no ✓/✗ — this tests real generalisation to a brand-new photo).

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    paths = [f'/content/{n}' for n in up]
    if paths:
        own = comparer.compare(paths, top_k=3, with_gradcam=True)
        viz.plot_model_comparison_panel(own, title='Uploaded image(s)');
except Exception as e:
    print('Upload unavailable here:', e)

---
### ⚠️ When finished: disconnect and DELETE the runtime
`Runtime > Disconnect and delete runtime` — Colab compute units are consumed the whole time a runtime is connected.